## Conclusions

**Model Comparison Insights:**

- **Faster R-CNN**: Slower but more accurate for object detection tasks
- **YOLOv8**: Fast and suitable for real-time detection
- **Confidence Scores**: Both models show different score distributions
- **Use Cases**: 
  - YOLOv8 for real-time applications
  - Faster R-CNN for high-accuracy offline analysis

In [ ]:
detection_counts = {
    'Image': [],
    'Faster R-CNN': [],
    'YOLOv8': []
}

for filename, image in test_images:
    rcnn_pred = predict_rcnn(rcnn_model, image, CONFIDENCE_THRESHOLD)
    yolo_pred = predict_yolo(yolo_model, image, CONFIDENCE_THRESHOLD)
    
    detection_counts['Image'].append(filename[:20])  # Truncate for display
    detection_counts['Faster R-CNN'].append(len(rcnn_pred['boxes']))
    detection_counts['YOLOv8'].append(len(yolo_pred['boxes']))

df_counts = pd.DataFrame(detection_counts)

fig, ax = plt.subplots(figsize=(12, 6))
x = np.arange(len(df_counts))
width = 0.35

bars1 = ax.bar(x - width/2, df_counts['Faster R-CNN'], width, label='Faster R-CNN', color='blue', alpha=0.8)
bars2 = ax.bar(x + width/2, df_counts['YOLOv8'], width, label='YOLOv8', color='green', alpha=0.8)

ax.set_xlabel('Image')
ax.set_ylabel('Detection Count')
ax.set_title('Detection Count Comparison Across Images')
ax.set_xticks(x)
ax.set_xticklabels(df_counts['Image'], rotation=45, ha='right')
ax.legend()
ax.grid(axis='y', alpha=0.3)

# Add value labels on bars
for bars in [bars1, bars2]:
    for bar in bars:
        height = bar.get_height()
        ax.text(bar.get_x() + bar.get_width()/2., height,
                f'{int(height)}', ha='center', va='bottom', fontsize=9)

plt.tight_layout()
plt.show()

print(f"\nAverage detections per image:")
print(f"  Faster R-CNN: {df_counts['Faster R-CNN'].mean():.1f}")
print(f"  YOLOv8: {df_counts['YOLOv8'].mean():.1f}")

## Detection Count Per Image

Track how many detections each model makes on individual images.

In [ ]:
import time

# Collect metrics
metrics = {
    'Model': [],
    'Total Detections': [],
    'Avg Confidence': [],
    'Inference Time (ms)': [],
    'FPS': []
}

for filename, image in test_images:
    # RCNN
    start = time.time()
    rcnn_pred = predict_rcnn(rcnn_model, image, CONFIDENCE_THRESHOLD)
    rcnn_time = (time.time() - start) * 1000
    
    # YOLOv8
    start = time.time()
    yolo_pred = predict_yolo(yolo_model, image, CONFIDENCE_THRESHOLD)
    yolo_time = (time.time() - start) * 1000

# Summary table
summary_data = {
    'Metric': ['Total Detections (threshold >= 0.5)', 'Mean Confidence Score', 
               'Median Confidence Score', 'Std Dev Confidence Score',
               'Min/Max Confidence Score'],
    'Faster R-CNN': [
        len([s for s in all_rcnn_scores if s >= CONFIDENCE_THRESHOLD]),
        f"{np.mean(all_rcnn_scores):.4f}",
        f"{np.median(all_rcnn_scores):.4f}",
        f"{np.std(all_rcnn_scores):.4f}",
        f"{np.min(all_rcnn_scores):.4f} / {np.max(all_rcnn_scores):.4f}"
    ],
    'YOLOv8': [
        len([s for s in all_yolo_scores if s >= CONFIDENCE_THRESHOLD]),
        f"{np.mean(all_yolo_scores):.4f}",
        f"{np.median(all_yolo_scores):.4f}",
        f"{np.std(all_yolo_scores):.4f}",
        f"{np.min(all_yolo_scores):.4f} / {np.max(all_yolo_scores):.4f}"
    ]
}

df_summary = pd.DataFrame(summary_data)
print("\n" + "="*70)
print("MODEL COMPARISON SUMMARY")
print("="*70)
print(df_summary.to_string(index=False))
print("="*70 + "\n")

## Performance Metrics Summary

In [ ]:
all_rcnn_scores = []
all_yolo_scores = []

for filename, image in test_images:
    rcnn_pred = predict_rcnn(rcnn_model, image, threshold=0.1)  # Low threshold for all scores
    yolo_pred = predict_yolo(yolo_model, image, threshold=0.1)
    
    all_rcnn_scores.extend(rcnn_pred["scores"])
    all_yolo_scores.extend(yolo_pred["scores"])

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# RCNN scores
axes[0].hist(all_rcnn_scores, bins=30, alpha=0.7, color='blue', edgecolor='black')
axes[0].axvline(CONFIDENCE_THRESHOLD, color='red', linestyle='--', linewidth=2, label=f'Threshold ({CONFIDENCE_THRESHOLD})')
axes[0].set_xlabel('Confidence Score')
axes[0].set_ylabel('Frequency')
axes[0].set_title('Faster R-CNN: Confidence Score Distribution')
axes[0].legend()
axes[0].grid(alpha=0.3)

# YOLOv8 scores
axes[1].hist(all_yolo_scores, bins=30, alpha=0.7, color='green', edgecolor='black')
axes[1].axvline(CONFIDENCE_THRESHOLD, color='red', linestyle='--', linewidth=2, label=f'Threshold ({CONFIDENCE_THRESHOLD})')
axes[1].set_xlabel('Confidence Score')
axes[1].set_ylabel('Frequency')
axes[1].set_title('YOLOv8: Confidence Score Distribution')
axes[1].legend()
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.show()

print(f"Faster R-CNN: {len(all_rcnn_scores)} total detections, mean score: {np.mean(all_rcnn_scores):.3f}")
print(f"YOLOv8: {len(all_yolo_scores)} total detections, mean score: {np.mean(all_yolo_scores):.3f}")

## Confidence Score Distribution

In [ ]:
def visualize_predictions(image: np.ndarray, rcnn_pred: Dict, yolo_pred: Dict, title: str = ""):
    """Draw predictions side-by-side"""
    
    fig, axes = plt.subplots(1, 2, figsize=(16, 6))
    fig.suptitle(f"Comparison: {title}", fontsize=14, fontweight='bold')
    
    # Original image size
    orig_h, orig_w = image.shape[:2]
    
    for idx, (ax, pred) in enumerate([(axes[0], rcnn_pred), (axes[1], yolo_pred)]):
        ax.imshow(image)
        ax.set_title(f"{pred['model']} ({len(pred['boxes'])} detections)", fontsize=12, fontweight='bold')
        ax.axis('off')
        
        # Draw boxes
        colors = ['lime', 'red']
        for box, score, label in zip(pred["boxes"], pred["scores"], pred["labels"]):
            x1, y1, x2, y2 = box
            
            # For RCNN, scale from 800x800 to original
            if pred['model'] == 'Faster R-CNN':
                scale_x = orig_w / IMG_SIZE[0]
                scale_y = orig_h / IMG_SIZE[1]
                x1, y1, x2, y2 = x1*scale_x, y1*scale_y, x2*scale_x, y2*scale_y
            
            w, h = x2 - x1, y2 - y1
            rect = patches.Rectangle((x1, y1), w, h, linewidth=2, 
                                    edgecolor=colors[label], facecolor='none')
            ax.add_patch(rect)
            
            # Add score label
            ax.text(x1, y1-5, f'{score:.2f}', fontsize=9, 
                   color=colors[label], bbox=dict(facecolor='black', alpha=0.5))
    
    plt.tight_layout()
    plt.show()

# Visualize first 5 images
for i, (filename, image) in enumerate(test_images[:5]):
    rcnn_pred = predict_rcnn(rcnn_model, image, CONFIDENCE_THRESHOLD)
    yolo_pred = predict_yolo(yolo_model, image, CONFIDENCE_THRESHOLD)
    visualize_predictions(image, rcnn_pred, yolo_pred, filename)

## Side-by-Side Prediction Comparison

Visualize predictions from both models on the same images.

In [ ]:
def predict_rcnn(model, image: np.ndarray, threshold: float = 0.5) -> Dict:
    """Predict with Faster R-CNN"""
    if model is None:
        return {"boxes": [], "scores": [], "labels": []}
    
    # Resize image
    img_resized = cv2.resize(image, IMG_SIZE)
    img_tensor = torch.from_numpy(img_resized).permute(2, 0, 1).float().div(255.0).to(DEVICE)
    
    with torch.no_grad():
        outputs = model([img_tensor])
    
    output = outputs[0]
    boxes = output["boxes"].cpu().numpy()
    scores = output["scores"].cpu().numpy()
    labels = output["labels"].cpu().numpy()
    
    # Filter by confidence
    mask = scores >= threshold
    return {
        "boxes": boxes[mask],
        "scores": scores[mask],
        "labels": labels[mask],
        "model": "Faster R-CNN"
    }

def predict_yolo(model, image: np.ndarray, threshold: float = 0.5) -> Dict:
    """Predict with YOLOv8"""
    if model is None:
        return {"boxes": [], "scores": [], "labels": []}
    
    # YOLOv8 expects BGR
    img_bgr = cv2.cvtColor(image, cv2.COLOR_RGB2BGR)
    results = model.predict(img_bgr, conf=threshold, verbose=False)
    
    if len(results) == 0 or len(results[0].boxes) == 0:
        return {"boxes": [], "scores": [], "labels": []}
    
    result = results[0]
    boxes = result.boxes.xyxy.cpu().numpy()
    scores = result.boxes.conf.cpu().numpy()
    labels = result.boxes.cls.cpu().numpy().astype(int)
    
    return {
        "boxes": boxes,
        "scores": scores,
        "labels": labels,
        "model": "YOLOv8"
    }

def scale_boxes(boxes: np.ndarray, from_size: Tuple, to_size: Tuple) -> np.ndarray:
    """Scale bounding boxes from one size to another"""
    if len(boxes) == 0:
        return boxes
    
    scale_x = to_size[0] / from_size[0]
    scale_y = to_size[1] / from_size[1]
    
    scaled = boxes.copy()
    scaled[:, [0, 2]] *= scale_x
    scaled[:, [1, 3]] *= scale_y
    return scaled

print("✓ Prediction functions defined")

## Prediction Functions

In [ ]:
import torchvision
from torchvision.models.detection.faster_rcnn import FastRCNNPredictor
from ultralytics import YOLO

# Configuration
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
IMG_SIZE = (800, 800)
CONFIDENCE_THRESHOLD = 0.5
TEST_IMAGE_DIR = "dataset/images/test"
RCNN_CHECKPOINT = "outputs/best_fasterrcnn.pth"
YOLO_CHECKPOINT = "runs_yolo8/weeds_yolov8s/weights/best.pt"

print(f"Device: {DEVICE}")
print(f"Test images: {TEST_IMAGE_DIR}")

# Load Faster R-CNN
def load_faster_rcnn():
    model = torchvision.models.detection.fasterrcnn_mobilenet_v3_large_fpn(weights=None)
    in_feats = model.roi_heads.box_predictor.cls_score.in_features
    model.roi_heads.box_predictor = FastRCNNPredictor(in_feats, 2)
    
    if os.path.exists(RCNN_CHECKPOINT):
        model.load_state_dict(torch.load(RCNN_CHECKPOINT, map_location=DEVICE))
        print(f"✓ Loaded Faster R-CNN from {RCNN_CHECKPOINT}")
    else:
        print(f"⚠ Faster R-CNN checkpoint not found: {RCNN_CHECKPOINT}")
    
    model.to(DEVICE).eval()
    return model

# Load YOLOv8
def load_yolov8():
    if os.path.exists(YOLO_CHECKPOINT):
        model = YOLO(YOLO_CHECKPOINT)
        print(f"✓ Loaded YOLOv8 from {YOLO_CHECKPOINT}")
        return model
    else:
        print(f"⚠ YOLOv8 checkpoint not found: {YOLO_CHECKPOINT}")
        return None

# Load test images
def load_test_images(max_images: int = 10) -> List[Tuple[str, np.ndarray]]:
    images = []
    if not os.path.exists(TEST_IMAGE_DIR):
        print(f"⚠ Test image directory not found: {TEST_IMAGE_DIR}")
        return images
    
    image_files = sorted([f for f in os.listdir(TEST_IMAGE_DIR) 
                         if f.lower().endswith(('.jpg', '.png', '.jpeg'))])[:max_images]
    
    for filename in image_files:
        img_path = os.path.join(TEST_IMAGE_DIR, filename)
        img = cv2.imread(img_path)
        if img is not None:
            img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
            images.append((filename, img_rgb))
    
    print(f"✓ Loaded {len(images)} test images")
    return images

# Initialize models and data
rcnn_model = load_faster_rcnn()
yolo_model = load_yolov8()
test_images = load_test_images(max_images=10)

## Setup: Load Models and Data

In [ ]:
import os
import torch
import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from pathlib import Path
from typing import Dict, List, Tuple
import warnings
warnings.filterwarnings('ignore')

# Ensure reproducibility
np.random.seed(42)
torch.manual_seed(42)

print("✓ All imports successful")

# Weed Detection Model Comparison Dashboard

This notebook compares predictions from **Faster R-CNN** and **YOLOv8** models on test images, providing detailed visualizations and performance metrics.